# JEPA-to-CLIP Embedding Pre-computation

Downloads MSR-VTT videos, runs V-JEPA and CLIP inference, saves paired embeddings to HDF5.

**Run on a GPU runtime** (Colab T4/A100 or similar). Downloads ~6GB data + ~3GB models.

In [ ]:
!pip install -q torch torchvision transformers h5py av tqdm Pillow huggingface_hub

In [ ]:
import torch
print(f"torch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    DEVICE = "cuda"
else:
    print("No GPU — will be slow but functional")
    DEVICE = "cpu"

## 1. Download MSR-VTT

In [ ]:
import os
from huggingface_hub import hf_hub_download

DATA_DIR = "data"
os.makedirs(f"{DATA_DIR}/videos", exist_ok=True)

# Download the video archive (~4.3GB)
zip_path = hf_hub_download(
    repo_id="AlexZigma/msr-vtt",
    filename="data/MSR-VTT.ZIP",
    repo_type="dataset",
    local_dir=DATA_DIR,
)
print(f"Downloaded to: {zip_path}")

In [ ]:
import zipfile

# Extract videos
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(DATA_DIR)

# Symlink videos to expected location
video_src = os.path.join(DATA_DIR, "MSR-VTT", "train_val_videos", "TrainValVideo")
video_dst = os.path.join(DATA_DIR, "videos")

# Move mp4 files into videos/ dir
import shutil
for f in os.listdir(video_src):
    if f.endswith(".mp4"):
        src = os.path.join(video_src, f)
        dst = os.path.join(video_dst, f)
        if not os.path.exists(dst):
            shutil.move(src, dst)

# Copy annotations
ann_src = os.path.join(DATA_DIR, "MSR-VTT", "train_val_annotation", "train_val_videodatainfo.json")
ann_dst = os.path.join(DATA_DIR, "annotations.json")
shutil.copy2(ann_src, ann_dst)

num_videos = len([f for f in os.listdir(video_dst) if f.endswith(".mp4")])
print(f"Extracted {num_videos} videos + annotations")

## 2. Configuration

In [ ]:
CONFIG = {
    "vjepa_model": "facebook/vjepa2-vitl-fpc64-256",
    "clip_model": "openai/clip-vit-large-patch14",
    "subset_size": 500,        # number of videos to process (set to None for all)
    "max_frames": 64,          # frames per video for V-JEPA
    "clip_sample_frames": 8,   # frames to sample for CLIP image encoder
    "max_captions": 5,         # captions per video to encode
    "output_path": "msrvtt_embeddings.h5",
    "device": DEVICE,
}
print(f"Will process {CONFIG['subset_size'] or 'ALL'} videos on {CONFIG['device']}")

## 3. Load Annotations & Parse Captions

In [ ]:
import json

with open(os.path.join(DATA_DIR, "annotations.json")) as f:
    ann_data = json.load(f)

videos = ann_data["videos"]
captions = {}
for sent in ann_data["sentences"]:
    vid_id = sent["video_id"]
    captions.setdefault(vid_id, []).append(sent["caption"])

# Filter to videos that exist locally
available_ids = set()
for f_name in os.listdir(os.path.join(DATA_DIR, "videos")):
    if f_name.endswith(".mp4"):
        available_ids.add(f_name.replace(".mp4", ""))

video_ids = [v["video_id"] for v in videos if v["video_id"] in available_ids]
if CONFIG["subset_size"]:
    video_ids = video_ids[:CONFIG["subset_size"]]

print(f"Available: {len(available_ids)} videos")
print(f"Processing: {len(video_ids)} videos")
print(f"Sample captions for {video_ids[0]}: {captions[video_ids[0]][:2]}")

## 4. Load Models

In [ ]:
from transformers import AutoModel, AutoVideoProcessor, CLIPModel, CLIPProcessor

device = torch.device(CONFIG["device"])

# V-JEPA (1024-dim embeddings)
print("Loading V-JEPA...")
vjepa_processor = AutoVideoProcessor.from_pretrained(CONFIG["vjepa_model"])
vjepa_model = AutoModel.from_pretrained(CONFIG["vjepa_model"], torch_dtype=torch.float16)
vjepa_model = vjepa_model.to(device).eval()
for p in vjepa_model.parameters():
    p.requires_grad = False
vjepa_dim = vjepa_model.config.hidden_size
print(f"V-JEPA loaded: {sum(p.numel() for p in vjepa_model.parameters())/1e6:.0f}M params, {vjepa_dim}-dim")

# CLIP (768-dim embeddings)
print("Loading CLIP...")
clip_processor = CLIPProcessor.from_pretrained(CONFIG["clip_model"])
clip_model = CLIPModel.from_pretrained(CONFIG["clip_model"], torch_dtype=torch.float16)
clip_model = clip_model.to(device).eval()
for p in clip_model.parameters():
    p.requires_grad = False
clip_dim = clip_model.config.projection_dim
print(f"CLIP loaded: {sum(p.numel() for p in clip_model.parameters())/1e6:.0f}M params, {clip_dim}-dim")

if torch.cuda.is_available():
    print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f}GB")

## 5. Define Embedding Functions

In [ ]:
import numpy as np
import torch.nn.functional as F
from PIL import Image
import av


def load_video_frames(video_path, max_frames=64):
    """Load uniformly-sampled RGB frames from a video file."""
    frames = []
    try:
        container = av.open(video_path)
        stream = container.streams.video[0]
        total = stream.frames or 0

        if total > 0 and total > max_frames:
            indices = set(np.linspace(0, total - 1, max_frames, dtype=int))
        else:
            indices = None

        for idx, frame in enumerate(container.decode(video=0)):
            if indices is not None and idx not in indices:
                continue
            frames.append(frame.to_ndarray(format="rgb24"))
            if len(frames) >= max_frames:
                break
        container.close()
    except Exception as e:
        print(f"Error loading {video_path}: {e}")
        return None

    return frames if len(frames) >= 8 else None


@torch.no_grad()
def compute_vjepa_embedding(frames):
    """V-JEPA embedding from video frames. Returns (1024,) float32 tensor."""
    inputs = vjepa_processor(frames, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = vjepa_model(**inputs)
    emb = outputs.last_hidden_state.squeeze(0).mean(dim=0)
    return emb.float().cpu()


@torch.no_grad()
def compute_clip_image_embedding(frames, sample_frames=8):
    """CLIP image embedding averaged over sampled frames. Returns (768,) float32 tensor."""
    n = len(frames)
    indices = np.linspace(0, n - 1, min(sample_frames, n), dtype=int)
    sampled = [frames[i] for i in indices]

    embeddings = []
    for frame in sampled:
        img = Image.fromarray(frame)
        inputs = clip_processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        emb = clip_model.get_image_features(**inputs)
        embeddings.append(emb.squeeze(0))

    avg = torch.stack(embeddings).mean(dim=0)
    return F.normalize(avg, dim=0).float().cpu()


@torch.no_grad()
def compute_clip_text_embeddings(caption_list):
    """CLIP text embeddings for captions. Returns (num_captions, 768) float32 tensor."""
    inputs = clip_processor(text=caption_list, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    emb = clip_model.get_text_features(**inputs)
    return F.normalize(emb, dim=-1).float().cpu()


# Quick sanity check
test_frames = load_video_frames(os.path.join(DATA_DIR, "videos", f"{video_ids[0]}.mp4"), max_frames=16)
if test_frames:
    print(f"Loaded {len(test_frames)} test frames, shape: {test_frames[0].shape}")
    test_jepa = compute_vjepa_embedding(test_frames)
    test_clip = compute_clip_image_embedding(test_frames)
    print(f"V-JEPA embedding: {test_jepa.shape}, CLIP embedding: {test_clip.shape}")
    print("Sanity check passed!")

## 6. Compute All Embeddings

This is the main loop. Progress bar shows per-video status with timing estimates.

In [ ]:
from tqdm.auto import tqdm
import time

max_captions = CONFIG["max_captions"]
max_frames = CONFIG["max_frames"]
clip_sample = CONFIG["clip_sample_frames"]

jepa_list = []
clip_image_list = []
clip_text_list = []
valid_ids = []
skipped = 0

pbar = tqdm(video_ids, desc="Computing embeddings")
for vid_id in pbar:
    video_path = os.path.join(DATA_DIR, "videos", f"{vid_id}.mp4")
    frames = load_video_frames(video_path, max_frames=max_frames)
    if frames is None:
        skipped += 1
        continue

    vid_caps = captions.get(vid_id, [])
    if not vid_caps:
        skipped += 1
        continue

    # Pad/truncate captions
    vid_caps = vid_caps[:max_captions]
    while len(vid_caps) < max_captions:
        vid_caps.append(vid_caps[-1])

    t0 = time.time()
    jepa_emb = compute_vjepa_embedding(frames)
    clip_img_emb = compute_clip_image_embedding(frames, sample_frames=clip_sample)
    clip_txt_emb = compute_clip_text_embeddings(vid_caps)
    dt = time.time() - t0

    jepa_list.append(jepa_emb)
    clip_image_list.append(clip_img_emb)
    clip_text_list.append(clip_txt_emb)
    valid_ids.append(vid_id)

    pbar.set_postfix({"done": len(valid_ids), "skipped": skipped, "last": f"{dt:.1f}s"})

print(f"\nProcessed {len(valid_ids)} videos, skipped {skipped}")

## 7. Save to HDF5

Saves all embeddings with train/val/test splits compatible with our local training pipeline.

In [ ]:
import h5py

# Stack into arrays
jepa_arr = torch.stack(jepa_list).numpy()       # (N, 1024)
clip_img_arr = torch.stack(clip_image_list).numpy()  # (N, 768)
clip_txt_arr = torch.stack(clip_text_list).numpy()   # (N, max_captions, 768)

N = len(valid_ids)
print(f"Total samples: {N}")
print(f"V-JEPA shape: {jepa_arr.shape}")
print(f"CLIP image shape: {clip_img_arr.shape}")
print(f"CLIP text shape: {clip_txt_arr.shape}")

# Split: 70% train, 15% val, 15% test
train_end = int(N * 0.70)
val_end = int(N * 0.85)

splits = {
    "train": (0, train_end),
    "val": (train_end, val_end),
    "test": (val_end, N),
}

output_path = CONFIG["output_path"]
with h5py.File(output_path, "w") as f:
    for split_name, (start, end) in splits.items():
        grp = f.create_group(split_name)
        grp.create_dataset("vjepa", data=jepa_arr[start:end])
        grp.create_dataset("clip_image", data=clip_img_arr[start:end])
        grp.create_dataset("clip_text", data=clip_txt_arr[start:end])
        grp.create_dataset("video_ids", data=[vid.encode() for vid in valid_ids[start:end]])
        print(f"  {split_name}: {end - start} samples")

file_size = os.path.getsize(output_path) / 1e6
print(f"\nSaved to {output_path} ({file_size:.1f} MB)")

## 8. Verify & Summary

Quick sanity check on saved embeddings + download instructions.

In [ ]:
# Verify the saved file
with h5py.File(output_path, "r") as f:
    print("HDF5 contents:")
    for split in ["train", "val", "test"]:
        grp = f[split]
        n = grp["vjepa"].shape[0]
        print(f"  {split}: {n} samples")
        print(f"    vjepa:      {grp['vjepa'].shape} {grp['vjepa'].dtype}")
        print(f"    clip_image: {grp['clip_image'].shape} {grp['clip_image'].dtype}")
        print(f"    clip_text:  {grp['clip_text'].shape} {grp['clip_text'].dtype}")
        
        # Check embedding norms (should be ~1.0 for CLIP, variable for V-JEPA)
        jepa_norms = np.linalg.norm(grp["vjepa"][:5], axis=1)
        clip_norms = np.linalg.norm(grp["clip_image"][:5], axis=1)
        print(f"    vjepa norms (first 5):  {jepa_norms.round(3)}")
        print(f"    clip norms (first 5):   {clip_norms.round(3)}")

print(f"\n--- DONE ---")
print(f"File: {output_path} ({os.path.getsize(output_path)/1e6:.1f} MB)")
print(f"Total: {N} video embeddings with paired CLIP image + text embeddings")
print(f"\nTo use locally, download this file and place it at:")
print(f"  LOGOS/PoCs/jepa_clip_translator/data/msrvtt_embeddings.h5")
print(f"\nThen run training:")
print(f"  python train.py --embeddings data/msrvtt_embeddings.h5 --verbose")

In [ ]:
# Download the embeddings file (Colab only)
try:
    from google.colab import files
    files.download(output_path)
    print("Download started!")
except ImportError:
    print("Not running in Colab — copy the file manually:")
    print(f"  scp <host>:{os.path.abspath(output_path)} LOGOS/PoCs/jepa_clip_translator/data/")